# Lesson 5 — The fee has three components and everyone models one

Every fill charges a trade fee ceiled to a hundredth of a cent, a rounding fee that restores the account's balance precision, and a rebate that returns accumulated rounding. On small fills the rounding fee can exceed the trade fee many times over.

**The rule.** `net_fee = ceil₀.₀₀₀₁(mult x rate x C x p x (1 - p)) + rounding − rebate`

**When it holds.** On every fill. The trade fee is a parabola peaking at fifty cents, so the correct no-arbitrage test is Σ ask < 1 − Σ net_fee, which depends on where the legs sit.

**When it fails.** Testing Σ ask < 1.00 is not a conservative approximation of that — it is a different test, and it is wrongest in the middle of the book where the volume is.

| | |
|---|---|
| Lesson id | `fees` |
| Pane it appears on | `fees` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/costs.py` |
| Tests that go red if it stops being true | `tests/test_coherence_costs.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The trade fee is a parabola, not a rate

In [ ]:
from modules.coherence.kernel.costs import (
    DEFAULT_TAKER_RATE,
    FeeSchedule,
    Fill,
    OrderFees,
    minimum_clip_hundredths,
    net_fee,
    no_arbitrage_bound,
    trade_fee,
)
from modules.coherence.kernel.money import contracts

SCHEDULE = FeeSchedule()
print(f"  published taker rate  {DEFAULT_TAKER_RATE}")
print(f"  series multiplier     {SCHEDULE.multiplier}   (it MULTIPLIES the rate, it is not the rate)")
print(f"  balance precision     {SCHEDULE.balance_precision}   (a hundred times finer for a direct member)")
print()
print("  price    trade fee on 100 contracts")
for cents in range(5, 100, 10):
    price = Decimal(cents) / 100
    print(f"  {price}     {trade_fee(Fill(price=price, size_hundredths=10_000), SCHEDULE)}")
print()
print("  A parabola peaking at fifty cents. It is the Bernoulli variance of the contract,")
print("  and it is why the fee-aware no-arbitrage test depends on WHERE the legs sit.")

## 2. The venue's own worked example, where rounding was the whole cost

In [ ]:
# Kalshi's own worked example: 0.09 contracts at $0.3301, filled in three lots.
order = OrderFees(schedule=FeeSchedule())
for lot in (3, 3, 3):
    piece = order.add(Fill(price=Decimal("0.3301"), size_hundredths=lot))
    print(f"  {lot} hundredths  trade {piece.trade_fee}  rounding {piece.rounding_fee}  rebate {piece.rebate}  net {piece.net}")

total = order.total
print()
print(f"  notional        {total.notional}")
print(f"  net fee         {total.net}")
print(f"  fee / notional  {total.as_fraction_of_notional.quantize(Decimal('0.0001'))}")
print()
print("  The first fill pays a trade fee of a twentieth of a cent against a rounding fee")
print("  nineteen times larger. The net fee exceeded the notional. Everyone models the")
print("  parabola; almost nobody models the component that dominated here.")

## 3. What fragmentation actually costs, measured

In [ ]:
print("  20 contracts at $0.45, filled in different numbers of pieces:")
for pieces in (1, 3, 9, 100):
    breakdown = net_fee(Decimal("0.4500"), 2_000, SCHEDULE, fills=pieces)
    print(
        f"    {pieces:>3} fill(s)  trade {breakdown.trade_fee}  rounding {breakdown.rounding_fee}  "
        f"rebate {breakdown.rebate}  net {breakdown.net}"
    )
print()
print("  The received wisdom is that fragmentation is itself a cost, because each fill")
print("  pays its own rounding. Run the model and that is very nearly false: the rebate")
print("  accumulator grows exactly as the rounding does and they cancel. What survives is")
print("  a residual bounded by the one cent the accumulator never returns.")

## 4. The minimum economic clip size, derived rather than guessed

In [ ]:
print("  smallest size at which an edge survives its own fees, assuming three fills:")
print()
print("  leg price   per-contract trade fee   edge 0.0500   edge 0.0200   edge 0.0050")
for leg_price in ("0.0500", "0.4500"):
    price = Decimal(leg_price)
    per_contract = trade_fee(Fill(price=price, size_hundredths=100), SCHEDULE)
    answers = []
    for edge_dollars in ("0.0500", "0.0200", "0.0050"):
        clip = minimum_clip_hundredths(price, Decimal(edge_dollars), SCHEDULE, expected_fills=3)
        answers.append("never" if clip is None else f"{contracts(clip)}")
    print(f"  {price}      {per_contract}                 " + "         ".join(f"{answer:>5}" for answer in answers))
print()
print("  Two things fall out. The minimum clip depends on WHERE the leg sits, because the")
print("  fee it has to clear is a parabola in the price. And an edge below the per-contract")
print("  fee never clears at any size — there is no clip that rescues it.")
print()
print("  Searched rather than solved: the fee has a ceiling and a floor in it, so it is a")
print("  step function of size and a closed form would be a fiction that happens to agree")
print("  at the tested points.")

## 5. The threshold a basket really has to beat

In [ ]:
for label, legs in (
    ("three legs near the middle", [Decimal("0.3300")] * 3),
    ("three legs in the tails   ", [Decimal("0.0200"), Decimal("0.0300"), Decimal("0.9400")]),
):
    bound = no_arbitrage_bound(legs, SCHEDULE)
    naive = Decimal("1.0000")
    print(f"  {label}  sum of asks {sum(legs, Decimal(0))}")
    print(f"    naive threshold  {naive}")
    print(f"    real threshold   {bound.quantize(Decimal('0.000001'))}")
    print(f"    the gap the naive test invents: {(naive - bound).quantize(Decimal('0.000001'))}")
print()
print("  Both baskets cost exactly the same 0.9900 and clear the naive test by the same")
print("  cent. One of them is a trade and the other is not, and only the fee model knows")
print("  which. Testing sum(ask) < 1.00 is not a conservative version of the fee-aware")
print("  test: it is a different test, wrongest in the middle of the book, where the")
print("  volume is.")